In [ ]:
# ==============================================================================
# CELL 1: SETUP - Install necessary libraries
# ==============================================================================
!pip install unidecode
!pip install textblob
print("Libraries installed.")

In [ ]:
# ==============================================================================
# CELL 2: PHASE 2 - BERTweet Training and Prediction Generation (UPDATED)
# ==============================================================================
import pandas as pd
import numpy as np
import re
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import string
import unidecode
import torch.nn.functional as F
import os

# --- Configuration & Functions (No changes here, same as before) ---
KAGGLE_INPUT_PATH = '/kaggle/input/nlp-getting-started/'
KAGGLE_WORKING_PATH = '/kaggle/working/'
MODEL_NAME = 'vinai/bertweet-base'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
N_SPLITS = 5
RANDOM_STATE = 42

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def clean_text_phase2(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+', '@USER', text)
    text = unidecode.unidecode(text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

class TweetDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_len):
        self.texts, self.targets, self.tokenizer, self.max_len = texts, targets, tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, item):
        text, target = str(self.texts[item]), self.targets[item]
        encoding = self.tokenizer.encode_plus(text, add_special_tokens=True, max_length=self.max_len, return_token_type_ids=False, padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt')
        return {'text': text, 'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'targets': torch.tensor(target, dtype=torch.long)}

def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler):
    model.train()
    losses = []
    for d in data_loader:
        input_ids, attention_mask, targets = d["input_ids"].to(device), d["attention_mask"].to(device), d["targets"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, targets)
        losses.append(loss.item())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
    return np.mean(losses)

def eval_model(model, data_loader, device):
    model.eval()
    predictions, actuals, all_logits = [], [], []
    with torch.no_grad():
        for d in data_loader:
            input_ids, attention_mask, targets = d["input_ids"].to(device), d["attention_mask"].to(device), d["targets"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            actuals.extend(d["targets"].cpu().numpy())
            all_logits.append(outputs.logits.cpu())
    return f1_score(actuals, predictions), predictions, torch.cat(all_logits)

# --- Main Execution for Phase 2 ---
def run_phase2():
    print("=== Phase 2: BERTweet Training & OOF Generation ===")
    train_df = pd.read_csv(os.path.join(KAGGLE_INPUT_PATH, 'train.csv')).fillna('')
    test_df = pd.read_csv(os.path.join(KAGGLE_INPUT_PATH, 'test.csv')).fillna('')
    train_df['clean_text'] = train_df['text'].apply(clean_text_phase2)
    test_df['clean_text'] = test_df['text'].apply(clean_text_phase2)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, normalization=True)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof_preds, oof_probs = np.zeros(len(train_df)), np.zeros(len(train_df))
    test_logits_list = []

    # --- Training Loop (condensed for brevity, no logical change) ---
    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['target'])):
        print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
        train_fold_df, val_fold_df = train_df.iloc[train_idx], train_df.iloc[val_idx]
        train_dataset = TweetDataset(texts=train_fold_df.clean_text.to_numpy(), targets=train_fold_df.target.to_numpy(), tokenizer=tokenizer, max_len=MAX_LEN)
        val_dataset = TweetDataset(texts=val_fold_df.clean_text.to_numpy(), targets=val_fold_df.target.to_numpy(), tokenizer=tokenizer, max_len=MAX_LEN)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
        optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
        total_steps = len(train_loader) * EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
        loss_fn = torch.nn.CrossEntropyLoss().to(device)
        best_f1 = 0
        for epoch in range(EPOCHS):
            print(f"  Epoch {epoch + 1}/{EPOCHS}")
            train_loss = train_epoch(model, train_loader, loss_fn, optimizer, device, scheduler)
            val_f1, _, _ = eval_model(model, val_loader, device)
            print(f"  Train Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")
            if val_f1 > best_f1:
                best_f1 = val_f1
                torch.save(model.state_dict(), os.path.join(KAGGLE_WORKING_PATH, f'bertweet_fold_{fold+1}.pth'))
        model.load_state_dict(torch.load(os.path.join(KAGGLE_WORKING_PATH, f'bertweet_fold_{fold+1}.pth')))
        _, oof_fold_preds, oof_fold_logits = eval_model(model, val_loader, device)
        oof_preds[val_idx], oof_probs[val_idx] = oof_fold_preds, F.softmax(oof_fold_logits, dim=1)[:, 1].cpu().numpy()
        test_dataset = TweetDataset(texts=test_df.clean_text.to_numpy(), targets=np.zeros(len(test_df)), tokenizer=tokenizer, max_len=MAX_LEN)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        _, _, fold_test_logits = eval_model(model, test_loader, device)
        test_logits_list.append(fold_test_logits)
        del model, optimizer, scheduler; torch.cuda.empty_cache()

    print(f"\nOverall OOF F1 Score (BERTweet): {f1_score(train_df['target'], oof_preds):.4f}")

    # --- Save outputs for Phase 3 AND create submission ---
    oof_df = pd.DataFrame({'id': train_df['id'], 'oof_prob_disaster': oof_probs})
    oof_df.to_csv(os.path.join(KAGGLE_WORKING_PATH, 'phase2_oof_preds.csv'), index=False)
    print("OOF probabilities saved to phase2_oof_preds.csv")

    avg_test_logits = torch.mean(torch.stack(test_logits_list), dim=0)
    avg_test_probs = F.softmax(avg_test_logits, dim=1)[:, 1].numpy()
    test_probs_df = pd.DataFrame({'id': test_df['id'], 'test_prob_disaster': avg_test_probs})
    test_probs_df.to_csv(os.path.join(KAGGLE_WORKING_PATH, 'phase2_test_probs.csv'), index=False)
    print("Test probabilities saved to phase2_test_probs.csv")

    # >>> NEW: Create Phase 2 submission file <<<
    avg_test_preds = torch.argmax(avg_test_logits, dim=1).numpy()
    submission_df_p2 = pd.DataFrame({'id': test_df['id'], 'target': avg_test_preds})
    submission_df_p2.to_csv(os.path.join(KAGGLE_WORKING_PATH, 'phase2_bertweet_submission.csv'), index=False)
    print("Submission file created: phase2_bertweet_submission.csv")

# Run the function
run_phase2()

In [ ]:
# ==============================================================================
# CELL 3: PHASE 3 - Advanced Features & LightGBM Training (UPDATED)
# ==============================================================================
import pandas as pd
import numpy as np
import re
import string
import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer # <-- FIX 1: Added this import
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import lightgbm as lgb
from scipy.sparse import hstack, csr_matrix
from textblob import TextBlob
import warnings
warnings.filterwarnings('ignore')

# --- Preprocessing & Feature Engineering Functions (No changes here, same as before) ---
def clean_text_phase3(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+', '@USER', text)
    text = unidecode.unidecode(text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_advanced_features(df):
    features = pd.DataFrame(index=df.index)
    df['sentiment'] = df['clean_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
    df['subjectivity'] = df['clean_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
    features['word_count'] = df['clean_text'].apply(lambda x: len(x.split()))
    features['unique_word_count'] = df['clean_text'].apply(lambda x: len(set(x.split())))
    features['unique_word_ratio'] = features['unique_word_count'] / (features['word_count'] + 1e-6)
    features['char_count'] = df['clean_text'].apply(len)
    features['keyword_word_count'] = df['keyword'].apply(lambda x: len(str(x).split('%20')))
    text_lower = df['clean_text'].str.lower()
    disaster_keywords = {'wildfire': ['fire', 'flames', 'california', 'burn', 'wildfire', 'forest fire'], 'flood': ['flood', 'water', 'rain', 'drown', 'tsunami', 'monsoon'], 'earthquake': ['earthquake', 'quake', 'shaking', 'tremor'], 'storm': ['storm', 'hurricane', 'cyclone', 'tornado', 'typhoon', 'wind'], 'accident': ['crash', 'accident', 'derailment', 'collision'], 'attack': ['attack', 'bomb', 'shooting', 'blast', 'explosion', 'suicide bomb']}
    for dtype, words in disaster_keywords.items(): features[f'{dtype}_score'] = text_lower.str.count(r'\b(' + '|'.join(words) + r')\b')
    disaster_prone_locs = ['california', 'japan', 'texas', 'florida', 'philippines', 'indonesia', 'india', 'china', 'houston', 'new orleans', 'mumbai', 'tokyo']
    loc_pattern = r'\b(' + '|'.join(disaster_prone_locs) + r')\b'
    features['is_in_disaster_prone_location'] = (text_lower.str.contains(loc_pattern) | df['location'].str.lower().str.contains(loc_pattern)).astype(int)
    features['california_wildfire_interaction'] = ((text_lower.str.contains('california')) & (features['wildfire_score'] > 0)).astype(int)
    urgency_words = ['breaking', 'now', 'just in', 'urgent', 'breaking news', 'alert']
    features['urgency_word_count'] = text_lower.str.count(r'\b(' + '|'.join(urgency_words) + r')\b')
    return pd.concat([features, df[['sentiment', 'subjectivity']]], axis=1)

def extract_basic_features(df):
    features = pd.DataFrame(index=df.index)
    features['text_length'] = df['text'].str.len()
    features['capitals_ratio'] = df['text'].str.count('[A-Z]') / (features['text_length'] + 1)
    features['hashtag_count'] = df['text'].str.count('#')
    features['location_missing'] = (df['location'] == '').astype(int)
    features['keyword_missing'] = (df['keyword'] == '').astype(int)
    return features

# --- Main Execution for Phase 3 ---
def run_phase3():
    print("\n=== Phase 3: LightGBM Training with Advanced Features ===")
    train_df = pd.read_csv(os.path.join(KAGGLE_INPUT_PATH, 'train.csv')).fillna('')
    test_df = pd.read_csv(os.path.join(KAGGLE_INPUT_PATH, 'test.csv')).fillna('')
    train_df['clean_text'] = train_df['text'].apply(clean_text_phase3)
    test_df['clean_text'] = test_df['text'].apply(clean_text_phase3)
    print("Extracting features...")
    train_basic_feats = extract_basic_features(train_df); test_basic_feats = extract_basic_features(test_df)
    train_adv_feats = extract_advanced_features(train_df); test_adv_feats = extract_advanced_features(test_df)
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=5000)
    train_tfidf = vectorizer.fit_transform(train_df['clean_text']); test_tfidf = vectorizer.transform(test_df['clean_text'])
    oof_df = pd.read_csv(os.path.join(KAGGLE_WORKING_PATH, 'phase2_oof_preds.csv'))
    train_bert_feats = oof_df[['oof_prob_disaster']]
    test_bert_df = pd.read_csv(os.path.join(KAGGLE_WORKING_PATH, 'phase2_test_probs.csv'))
    test_bert_feats = test_bert_df[['test_prob_disaster']]
    print("Loaded BERTweet OOF features successfully.")
    X_train = hstack([train_tfidf, csr_matrix(train_basic_feats), csr_matrix(train_adv_feats), csr_matrix(train_bert_feats)]).tocsr()
    X_test = hstack([test_tfidf, csr_matrix(test_basic_feats), csr_matrix(test_adv_feats), csr_matrix(test_bert_feats)]).tocsr()
    y_train = train_df['target']
    print(f"Total features for LightGBM: {X_train.shape[1]}")
    lgb_params = {'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt', 'n_estimators': 10000, 'learning_rate': 0.01, 'num_leaves': 20, 'max_depth': 5, 'seed': RANDOM_STATE, 'n_jobs': -1, 'verbose': -1, 'colsample_bytree': 0.7, 'subsample': 0.7, 'reg_alpha': 0.1, 'reg_lambda': 0.1}
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof_preds_lgbm, test_preds_lgbm_prob = np.zeros(len(train_df)), np.zeros(len(test_df))

    # --- Training Loop (condensed for brevity, no logical change) ---
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
        X_train_fold, y_train_fold = X_train[train_idx], y_train[train_idx]
        X_val_fold, y_val_fold = X_train[val_idx], y_train[val_idx]
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)], eval_metric='f1', callbacks=[lgb.early_stopping(100, verbose=False)])
        val_preds_proba = model.predict_proba(X_val_fold)[:, 1]
        oof_preds_lgbm[val_idx] = (val_preds_proba > 0.5).astype(int)
        test_preds_lgbm_prob += model.predict_proba(X_test)[:, 1] / N_SPLITS

    print(f"\nOverall OOF F1 Score (LightGBM): {f1_score(y_train, oof_preds_lgbm):.4f}")
    lgbm_probs_df = pd.DataFrame({'id': test_df['id'], 'test_prob_disaster': test_preds_lgbm_prob})
    lgbm_probs_df.to_csv(os.path.join(KAGGLE_WORKING_PATH, 'phase3_test_probs.csv'), index=False)
    print("LGBM test probabilities saved to phase3_test_probs.csv")

    # >>> NEW: Create Phase 3 submission file <<<
    final_test_preds_p3 = (test_preds_lgbm_prob > 0.5).astype(int)
    submission_df_p3 = pd.DataFrame({'id': test_df['id'], 'target': final_test_preds_p3})
    submission_df_p3.to_csv(os.path.join(KAGGLE_WORKING_PATH, 'phase3_lgbm_submission.csv'), index=False)
    print("Submission file created: phase3_lgbm_submission.csv")

run_phase3()

In [ ]:
# ==============================================================================
# CELL 4: PHASE 4 - Final Ensemble and Submission (UPDATED)
# ==============================================================================
import pandas as pd
import os

print("\n=== Phase 4: Final Model Ensemble ===")

bert_probs_df = pd.read_csv(os.path.join(KAGGLE_WORKING_PATH, 'phase2_test_probs.csv'))
lgbm_probs_df = pd.read_csv(os.path.join(KAGGLE_WORKING_PATH, 'phase3_test_probs.csv'))

# Define weights - a good starting point is to give more weight to the stronger model (BERTweet)
bert_weight = 0.70
lgbm_weight = 0.30

# Calculate weighted average
final_probs = bert_weight * bert_probs_df['test_prob_disaster'] + lgbm_weight * lgbm_probs_df['test_prob_disaster']

# Generate final predictions using a 0.5 threshold
final_preds = (final_probs > 0.5).astype(int)

# Create the final submission file
# Using 'submission.csv' as the filename is standard practice for Kaggle notebooks.
submission_df_final = pd.read_csv(os.path.join(KAGGLE_INPUT_PATH, 'sample_submission.csv'))
submission_df_final['target'] = final_preds
submission_df_final.to_csv('submission.csv', index=False) 
# Also save with a more descriptive name
submission_df_final.to_csv(os.path.join(KAGGLE_WORKING_PATH, 'final_ensemble_submission.csv'), index=False)


print(f"\nEnsemble created with weights: BERTweet={bert_weight}, LGBM={lgbm_weight}")
print("Final submission file 'submission.csv' created successfully!")
print("You can now find all three submission files in the Output section of your notebook.")